# Q_C DataStructure demo

Give a **root**, pick which **streams** to load, get back the cross-session **trial table** (and, if DLC is on, the aligned pose, sliceable by trial). One `load()` call, one result dict.

## 1. Configure

Set the root and toggle streams on/off. `streams` may include any of:
`'events'` (always on, builds the trial table), `'nosepoke'`, `'soundcard'`, `'session_settings'`, `'video'`, `'dlc'`.

In [ ]:
from data_conduit.qc import qc_datastructure, slice_pose_for_trial
from pathlib import Path

ROOT = Path("/media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput/Training")
ds = qc_datastructure(
    ROOT,
    depth=2,                                   # firstdata layout: mouse / day / session
    level_names=('mouseID', 'day'),            # retained as provenance columns
    # streams=('events', 'dlc'),                 # <- toggle which streams to load
    trial_start_buffer=0,                      # no inter-trial buffer (unlike Q_C_Analysis_Workflow)
    l0_selector='FbR_M01569522',               # <- optional level filters (mouse, day, ...)
    # l1_selector='Day 11',
    # include=[...], exclude=[...],            # <- optional session allow/deny lists
)

list(ds.select())                              # which sessions matched

## 2. Load

Each value is one stream concatenated across all matched sessions, tagged per entry with `session` + each level (`mouseID`, `day`). A stream missing from some sessions (e.g. DLC) is kept for the ones that have it; a missing reader just warns.

In [2]:
result = ds.load()
list(result.keys())


                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  


/home/callum/Keshavarzi-Lab-Workspace/data-conduit/src/data_conduit/datastructure.py:536: UserWarning: reader 'dlc' skipped for session '2026-03-26T13-29-40': FileNotFoundError: no 'DLC' folder in session /home/callum/Keshavarzi-Lab-Workspace/data-conduit/datasets/firstdata/BonsaiFiles/FbR_M01569522/Day 01/2026-03-26T13-29-40.
  objects = self.catalog.read_session(info['path'])
/home/callum/Keshavarzi-Lab-Workspace/data-conduit/src/data_conduit/datastructure.py:536: UserWarning: reader 'dlc' skipped for session '2026-03-26T13-56-31': FileNotFoundError: no 'DLC' folder in session /home/callum/Keshavarzi-Lab-Workspace/data-conduit/datasets/firstdata/BonsaiFiles/FbR_M01569522/Day 01/2026-03-26T13-56-31.
  objects = self.catalog.read_session(info['path'])
/home/callum/Keshavarzi-Lab-Workspace/data-conduit/src/data_conduit/datastructure.py:536: UserWarning: reader 'dlc' skipped for session '2026-04-07T17-00-58': FileNotFoundError: no 'DLC' folder in session /home/callum/Keshavarzi-Lab-Works

['nosepoke:Activations',
 'nosepoke:LEDs',
 'nosepoke:Valves',
 'nosepoke:Rewards',
 'session_settings:metadata',
 'session_settings:trials',
 'video',
 'trials',
 'dlc:position',
 'dlc:confidence']

In [3]:
trials = result['trials']
trials.head()

,trial_index,start_time,end_time,tz_triggered_time,outbound_start_time,outbound_end_time,inbound_start_time,inbound_end_time,TTT,TTP,ChosenPort,CorrectPort,outcome,LED,angle_offset,target_zone_size,session,mouseID,day
0,1,6351.754976,6437.364832,6358.117984,6351.754976,6358.117984,6358.117984,6437.364832,6.363008,79.246848,1,0,Success,OFF,20.0,245.0,2026-03-26T13-29-40,FbR_M01569522,Day 01
1,2,6437.364832,6537.883488,6443.664992,6437.364832,6443.664992,6443.664992,6537.883488,6.300160,94.218496,5,0,Success,OFF,20.0,245.0,2026-03-26T13-29-40,FbR_M01569522,Day 01
2,3,6537.883488,6565.522976,6544.094976,6537.883488,6544.094976,6544.094976,6565.522976,6.211488,21.428000,17,0,Success,OFF,100.0,245.0,2026-03-26T13-29-40,FbR_M01569522,Day 01
3,4,6565.522976,6646.884992,6574.976992,6565.522976,6574.976992,6574.976992,6646.884992,9.454016,71.908000,10,0,Success,OFF,-20.0,245.0,2026-03-26T13-29-40,FbR_M01569522,Day 01
4,5,6646.884992,6666.685984,6658.224000,6646.884992,6658.224000,6658.224000,6666.685984,11.339008,8.461984,9,0,Success,OFF,-160.0,245.0,2026-03-26T13-29-40,FbR_M01569522,Day 01


## 3. Slice DLC pose by a trial's path windows

(Only when `'dlc'` is in `streams`.) Pick a trial and pull the pose for its `outbound`, `inbound`, or whole-`trial` window. Trial times and pose Time share the Bonsai Seconds clock, so the slice is just that session's pose inside the window.

In [ ]:
if 'dlc:position' not in result:
    # No DLC pose for the selected sessions (no 'DLC' folder on disk -> the dlc
    # reader was skipped for every session, so the key was never created).
    print("No DLC pose for the selected sessions - skipping the slice demo. "
          "Add 'dlc' to streams and run DeepLabCut on these sessions to enable it.")
else:
    pose = result['dlc:position']                      # (Time x keypoints x space)
    trial = trials[trials['tz_triggered_time'].notna()].iloc[0]

    outbound = slice_pose_for_trial(pose, trial, segment='outbound')
    inbound  = slice_pose_for_trial(pose, trial, segment='inbound')
    display(outbound.to_dataframe())